## Personalized Scoring

Each attraction $i$ receives a personalized score that combines how well it matches the user's interests with its general quality. The score is fixed per attraction, so the value of an itinerary is the sum of the scores of the attractions it visits. Travel and visiting times are not part of the score; they are handled by the search algorithms.

**User preferences.** The user rates each of the $K = 8$ categories (History, Art, Architecture, Nature, Local Culture, Views, Religion, Science) with $w_c \in \{0, 1, \dots, 5\}$. Let $x_{ic} \in \{0, 1\}$ indicate whether attraction $i$ belongs to category $c$.

**Interest match.** The average rating the user gave to the categories of the attraction, scaled to $[0, 1]$:

$$I_i = \frac{\sum_{c=1}^{K} w_c \, x_{ic}}{5 \sum_{c=1}^{K} x_{ic}}$$

Averaging over the attraction's own categories prevents attractions with many labels from being favored only because they have more labels.

**General quality.** The mean of the min-max normalized rating $R_i$ and importance $M_i$:

$$Q_i = \tfrac{1}{2}\,\mathrm{minmax}(R_i) + \tfrac{1}{2}\,\mathrm{minmax}(M_i), \qquad \mathrm{minmax}(z_i) = \frac{z_i - \min_j z_j}{\max_j z_j - \min_j z_j}$$

**Personalized score.**

$$s_i = 100 \cdot I_i \cdot \left(1 - \lambda + \lambda\, Q_i\right), \qquad \lambda \in [0, 1]$$

The product ensures that an attraction the user is not interested in receives a low score regardless of its popularity, while $\lambda$ controls how much general quality matters relative to personal interest ($\lambda = 0$ uses interest only). Since $I_i \in [0,1]$ and $Q_i \in [0,1]$, every score satisfies $0 \le s_i \le 100$, which keeps the scores non-negative and additive for the search heuristics.

In [1]:
import numpy as np
import pandas as pd

ATTRACTIONS_PATH = "input_data/Barcelona_20_Attractions_With_Importance.xlsx"
TIMES_PATH = "input_data/Barcelona_Times_20_Attractions.xlsx"

CATEGORIES = ["History", "Art", "Architecture", "Nature", "Local Culture", "Views", "Religion", "Science"]

attractions = pd.read_excel(ATTRACTIONS_PATH).set_index("Attraction")
walking_times = pd.read_excel(TIMES_PATH, sheet_name="Walking", index_col=0)
transit_times = pd.read_excel(TIMES_PATH, sheet_name="Public transport", index_col=0)


def minmax(values):
    values = np.asarray(values, dtype=float)
    return (values - values.min()) / (values.max() - values.min())


def personalized_scores(attractions, preferences, lam=0.5):
    missing = [c for c in CATEGORIES if c not in preferences]
    if missing:
        raise ValueError(f"Missing preferences for: {missing}")
    weights = np.array([preferences[c] for c in CATEGORIES], dtype=float)
    if weights.min() < 0 or weights.max() > 5:
        raise ValueError("Preferences must be between 0 and 5")
    if not 0 <= lam <= 1:
        raise ValueError("lam must be between 0 and 1")

    x = attractions[CATEGORIES].to_numpy(dtype=float)
    interest = (x @ weights) / (5 * x.sum(axis=1))
    quality = 0.5 * minmax(attractions["Rating /5"]) + 0.5 * minmax(attractions["Importance /5"])
    score = 100 * interest * (1 - lam + lam * quality)

    return pd.DataFrame(
        {"interest": interest, "quality": quality, "score": score},
        index=attractions.index,
    )


user_preferences = {
    "History": 5, "Art": 2, "Architecture": 5, "Nature": 1,
    "Local Culture": 2, "Views": 3, "Religion": 1, "Science": 1,
}

scores = personalized_scores(attractions, user_preferences, lam=0.5)
scores.sort_values("score", ascending=False).round(2)

,interest,quality,score
Attraction,,,
Casa Batlló,0.80,0.80,72.00
Sagrada Família,0.65,1.00,65.00
Recinte Modernista de Sant Pau,0.80,0.53,61.33
MNAC,0.75,0.53,57.50
Búnquers del Carmel,0.80,0.43,57.33
Museu Picasso,0.70,0.63,57.17
Catedral de Barcelona,0.73,0.53,56.22
Santa Maria del Mar,0.73,0.43,52.56
Park Güell,0.55,0.90,52.25
